# Lösung: Vom Training zu Vorhersagen

Arbeite [das Aufgaben-Notebook](../3_from_training_to_predictions.ipynb) durch, bevor du dir diese Musterlösung ansiehst. Die sechs Antworten unten sind eine mögliche Implementierung; andere explizite Eingaberichtlinien und Interpretationen können gültig sein.

Führ zuerst Notebook 1 bis einschließlich **Export für Notebook 3** aus. Diese Lösung lädt die dort gespeicherte Titanic-Pipeline in einem frisch gestarteten Kernel und braucht keine Datenbankverbindung. Sie enthält die angeleiteten Beispiele, damit sie unabhängig von den Variablen des Aufgaben-Notebooks im Arbeitsspeicher läuft. Hier wird kein Modell trainiert oder getunt.

Die neuen Passagiereinträge sind **erfundene Beispiele**, keine neuen Beobachtungen mit Labels. Sie zeigen den Umgang mit Eingaben und die Vorhersagen; sie können weder die Accuracy messen noch den Einsatz in der Praxis validieren.

## Lernziele

- Die vollständige trainierte Pipeline und ihre Metadaten speichern und neu laden.
- Training von Inferenz unterscheiden: die Anwendung eines vorhandenen Modells auf neue Eingaben.
- Vorhergesagte Klassen und Wahrscheinlichkeiten lesen, ohne sie als Garantien zu behandeln.
- Fehlende Werte, ungültige Werte und für das Modell ungewohnte Werte voneinander trennen.
- Einen Batch verarbeiten und dabei abgelehnte Zeilen und ihre Gründe behalten.
- Die Empfindlichkeit gegenüber Änderungen der Eingaben untersuchen und eine kurze Modellübergabe vorbereiten.

Jede Aufgabe unten enthält lauffähigen Code und eine Interpretation. Vergleich die Entscheidungen mit deiner eigenen Arbeit, statt anzunehmen, dass dies die einzige gültige Eingaberichtlinie ist.

## Vorhersage-Workflow

<img src="../assets/prediction_workflow.svg" width="800" style="max-width: 100%;" alt="Neue Einträge werden validiert; akzeptierte Einträge durchlaufen die gespeicherte Vorverarbeitung und das Modell; Vorhersagen und abgelehnte Zeilen sind im exportierten Batch-Report enthalten.">

Beim Training wurden die Transformationen und der Klassifikator gelernt. Die Inferenz verwendet beides wieder: Hier gibt es kein neues `fit`. Eine ungültige Eingabe ist ein Problem der Schnittstelle, auch wenn ein Modell dafür eine Zahl zurückgeben würde.

## 1. Gespeicherte Pipeline laden

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "solutions":
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
MODEL_PATH = ARTIFACT_DIR / "titanic_pipeline.joblib"
INFO_PATH = ARTIFACT_DIR / "titanic_model_info.json"
if not MODEL_PATH.is_file() or not INFO_PATH.is_file():
    raise FileNotFoundError(
        "Run notebook 1 through Export for Notebook 3, then reopen notebook 3 "
        "from the project root."
    )

Die JSON-Datei hält die Features, die gelernten Kategorien, die Paketversionen und die bereits in Notebook 1 durchgeführte Evaluation fest. Prüf die Paketversionen, bevor du das Modell lädst. Verwende dieselbe Projektumgebung (`uv sync`), mit der das Artefakt erstellt wurde.

Lade nur ein Artefakt, das du selbst erstellt hast oder dem du vertraust: `joblib.load` kann Python-Code ausführen. KNN behält außerdem Trainingsbeobachtungen im trainierten Modell, ein gespeichertes Modell ist also nicht unbedingt frei von Trainingsdaten. Diese Artefakte bleiben im von Git ignorierten Verzeichnis `artifacts/`.

[Referenz: scikit-learn model persistence](https://scikit-learn.org/stable/model_persistence.html).

In [ ]:
model_info = json.loads(INFO_PATH.read_text())
for package, saved_version in model_info["package_versions"].items():
    current_version = version(package)
    if current_version != saved_version:
        raise RuntimeError(
            f"{package}: artifact uses {saved_version}, environment uses "
            f"{current_version}. Recreate the artifact with the project environment."
        )

model = joblib.load(MODEL_PATH)
FEATURES = model_info["input_features"]
NUMERIC = model_info["numeric_features"]
CATEGORICAL = model_info["categorical_features"]
KNOWN_CATEGORIES = model_info["known_categories"]

pd.Series(
    {
        "Selected search": model_info["selected_search"],
        "Classifier": type(model.steps[-1][1]).__name__,
        "Features": ", ".join(FEATURES),
        "Recorded CV accuracy": model_info["evaluation"]["cv_accuracy"],
        "Recorded test accuracy": model_info["evaluation"]["test_accuracy"],
    },
    name="Saved model",
)

### Kontrolle: Was wurde gespeichert?

Erklär, warum es für die sechs rohen Features nicht ausreichen würde, nur den KNN-Klassifikator zu speichern. Welche Informationen bräuchtest du, um die numerischen und kategorialen Transformationen zu reproduzieren?

<details>
<summary>Erklärung anzeigen</summary>

Der Klassifikator erwartet transformierte Features. Die vollständige Pipeline behält zusätzlich die gefitteten Imputer, den Scaler, das Kategorienvokabular und die Spaltenzuordnung. Diese auf dem Vorhersage-Batch neu zu berechnen, würde die Darstellung verändern. Die beiden Accuracy-Werte oben dokumentieren eine frühere Evaluation; sie sind keine Scores für die erfundenen Einträge von heute.
</details>

## 2. Einen neuen Eintrag vorhersagen

`Survived = 1` ist die positive Klasse in diesem historischen Datensatz. `predict` gibt eine Klasse zurück; `predict_proba` gibt eine Spalte pro Klasse in `model.classes_` zurück. Ermittle die Spalte für Klasse 1 explizit, statt dich auf einen unerklärten Spaltenindex zu verlassen.

In [ ]:
new_passenger = pd.DataFrame(
    [{"Age": 28, "SibSp": 0, "Parch": 0, "Fare": 7.9, "Sex": "male", "Embarked": "S"}]
)
positive_column = list(model.classes_).index(1)
single_prediction = model.predict(new_passenger)
single_probability = model.predict_proba(new_passenger)[:, positive_column]
pd.DataFrame(
    {"predicted_class": single_prediction, "probability_class_1": single_probability}
)

KNN-Wahrscheinlichkeiten hängen von den Stimmen ab, die benachbarte Trainingsbeobachtungen für die Klassen abgeben (und von einer etwaigen Gewichtung nach Distanz). Sie sind nicht automatisch kalibrierte Wahrscheinlichkeiten. Bei einheitlichen Gewichten können sie sich in diskreten Schritten ändern. Ein Wert nahe 0,5 bedeutet, dass die Klassen ähnlich viele Stimmen erhalten; er sagt dir nicht, ob die Eingabe gültig oder repräsentativ ist.

Bei diesem binären KNN wird bei einem exakten Gleichstand der Wahrscheinlichkeiten die erste Klasse in `classes_` gewählt; eine manuelle Prüfung mit `probability >= 0.5` könnte deshalb von `predict` abweichen. Wir behalten die Vorhersageregel des trainierten Modells bei.

[Referenz: KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html).

### Aufgabe 1: eigene Vorhersage-Requests

Erstell mindestens vier erfundene Einträge mit eindeutigen Werten für `request_id`: einen gewöhnlichen Eintrag, einen mit einem fehlenden Messwert, einen mit einer Kategorie, die im Training nicht vorkam, und einen mit einem unmöglichen Wert. Verwende die sechs erwarteten Features und sag voraus, welche Einträge akzeptiert, zur Prüfung markiert oder abgelehnt werden sollten.

Heb diese Einträge für Aufgabe 6 auf. Verwende `model.predict` vorerst nur für den gewöhnlichen Eintrag direkt; der nächste Abschnitt ergänzt die Validierung für die anderen Fälle. Erklär, warum du ihre Accuracy ohne tatsächliche Labels nicht berechnen kannst.

In [ ]:
# Die Metadaten der Requests von den sechs Modelleingaben getrennt halten.
own_requests = pd.concat([new_passenger] * 4, ignore_index=True)
own_requests.insert(0, "request_id", ["own-01", "own-02", "own-03", "own-04"])
own_requests.loc[1, "Age"] = np.nan
own_requests.loc[2, "Embarked"] = "X"
own_requests.loc[3, "Age"] = -5

own_expectations = pd.DataFrame(
    {
        "request_id": own_requests["request_id"],
        "expected_status": ["predicted", "predicted", "review", "rejected"],
    }
)
ordinary_input = own_requests.loc[[0], FEATURES]
print(own_expectations.to_string(index=False))
pd.DataFrame(
    {
        "request_id": ["own-01"],
        "predicted_class": model.predict(ordinary_input),
        "probability_class_1": model.predict_proba(ordinary_input)[:, positive_column],
    }
)

Ein fehlendes Alter kann der gefittete Imputer füllen. Der unbekannte Hafen braucht eine Prüfung; ein negatives Alter wird abgelehnt. Diese Entscheidungen beschreiben den Input Contract, nicht, welche Vorhersagen richtig sind. Keiner dieser erfundenen Einträge hat ein tatsächliches Ergebnis, deshalb kann keine Accuracy berechnet werden.

## 3. Input Contract festlegen

Ein Input Contract legt fest, was der Vorhersage-Workflow akzeptiert. Die folgenden Regeln sind Konventionen für diese Aufgabe; sie beweisen nicht, dass sich das historische Modell für neue Populationen eignet.

| Eingabe | Vertrag |
|---|---|
| `request_id` | Vorhanden und innerhalb des Batches eindeutig; dient dazu, Ergebnisse den Requests zuzuordnen. |
| Sechs Feature-Spalten | Alle Spalten müssen vorhanden sein, auch wenn einzelne Werte fehlen. Zusätzliche Spalten werden abgelehnt, um Schemafehler zu finden. |
| `Age` | Zahl zwischen 0 und 120 oder fehlend. |
| `Fare` | Endliche, nicht negative Zahl oder fehlend; dieselben Einheiten wie in der Trainings-CSV. |
| `SibSp`, `Parch` | Endliche, nicht negative, ganzzahlige Anzahlen oder fehlend. |
| `Sex`, `Embarked` | Kategorien aus dem Trainingsvokabular sind vertraut. Ungewohnte Kategorien werden zur Prüfung markiert. Fehlende Werte füllt der gefittete Imputer. |

Fehlende Werte sind nicht dasselbe wie fehlende Spalten. Der Imputer kann einen fehlenden Messwert füllen, aber nicht die Bedeutung einer fehlenden Feature-Spalte rekonstruieren. Ein Wert außerhalb eines Wertebereichs der Trainingsdaten kann physikalisch gültig und dem Modell trotzdem ungewohnt sein.

In [ ]:
contract = pd.DataFrame(
    {
        "feature": FEATURES,
        "kind": ["numeric" if name in NUMERIC else "categorical" for name in FEATURES],
        "training_categories": [KNOWN_CATEGORIES.get(name, []) for name in FEATURES],
    }
)
contract

### Aufgabe 2: Triage vor der Vorhersage

Lade unten den bereitgestellten Batch. Bevor du den Validator ausführst, schreib eine kleine Entscheidungstabelle für seine Einträge: `predicted`, `review` oder `rejected`, mit einem Grund. Achte besonders auf ein fehlendes Alter, Alter 100, Alter -5, ein nicht numerisches Alter, eine nicht ganzzahlige Geschwisteranzahl, einen unbekannten Hafen und eine Zeile ohne Feature-Werte.

Entscheide, ob für einen Prüffall eine Vorhersage angezeigt werden soll, und erklär, wie du sie von einer gewöhnlichen Vorhersage unterscheiden würdest.

In [ ]:
batch = pd.read_csv(PROJECT_ROOT / "data" / "titanic_prediction_requests.csv")
batch

In [ ]:
expected_triage = pd.DataFrame(
    [
        ("request-01", "predicted", "Ordinary record"),
        ("request-02", "predicted", "Missing age uses the fitted imputer"),
        ("request-03", "review", "Port X was not observed in training"),
        ("request-04", "rejected", "Negative age"),
        ("request-05", "rejected", "Age is not numeric"),
        ("request-06", "review", "Age 100 exceeds the observed training range"),
        ("request-07", "rejected", "Fare is not numeric"),
        ("request-08", "rejected", "Sibling count is fractional"),
        ("request-09", "predicted", "Missing sex uses the fitted imputer"),
        ("request-10", "review", "Sex category absent from training"),
        ("request-11", "predicted", "Ordinary record"),
        ("request-12", "review", "Fare exceeds the observed training range"),
        ("request-13", "predicted", "Missing sibling count uses the fitted imputer"),
        ("request-14", "review", "Every feature is missing"),
    ],
    columns=["request_id", "expected_status", "explanation"],
)
expected_triage

Diese Musterlösung zeigt Vorhersagen für Prüffälle an, kennzeichnet sie aber als **review** und behält die Gründe. Eine Schnittstelle sollte diese Markierungen sichtbar halten, statt jede Wahrscheinlichkeit als gleich vertrauenswürdig darzustellen. Prüffälle stattdessen abzulehnen, kann eine gültige Richtlinie sein, wenn sie dokumentiert und konsistent implementiert wird.

## 4. Validieren ohne neues Training

Fang mit der Struktur des Batches an. Eine fehlende Spalte oder eine mehrdeutige Request-Kennung blockiert den ganzen Batch, weil wir die Eingaben dann nicht zuverlässig zusammenstellen und die Ausgaben nicht zuordnen können. Fehlerhafte Werte werden unten Zeile für Zeile behandelt, sodass ein einzelnes ungültiges Alter nicht verhindern muss, dass andere Einträge verarbeitet werden.

In [ ]:
def check_structure(records):
    """Prüft das Schema und die Kennungen, bevor die Feature-Werte untersucht werden."""
    expected = {"request_id", *FEATURES}
    if records.empty:
        raise ValueError("The batch must contain at least one request.")
    if not records.columns.is_unique:
        raise ValueError("Column names must be unique.")
    missing = expected - set(records.columns)
    extra = set(records.columns) - expected
    if missing or extra:
        raise ValueError(
            f"Missing columns: {sorted(missing)}; extra columns: {sorted(extra)}"
        )
    ids = records["request_id"]
    if (
        ids.isna().any()
        or ids.astype(str).str.strip().eq("").any()
        or ids.duplicated().any()
    ):
        raise ValueError("request_id must be present and unique within the batch.")


check_structure(batch)
print("Batch structure accepted.")

In [ ]:
# Eine fehlende Spalte ist ein Schemafehler, kein Messwert, den der Imputer füllen soll.
try:
    check_structure(batch.drop(columns="Age"))
except ValueError as error:
    print("Expected rejection:", error)

Konvertiere numerische Eingaben explizit, halte aber jeden Konvertierungsfehler fest. `errors="coerce"` unten ist nur deshalb sinnvoll, weil fehlgeschlagene Konvertierungen anschließend abgelehnt werden; das dabei entstehende NaN wie einen gewöhnlichen fehlenden Messwert zu behandeln, würde ungültige Eingaben verbergen.

In [ ]:
def prepare_inputs(records):
    """Vereinheitlicht die numerischen Datentypen, ohne neue Kennzahlen für die Vorverarbeitung zu lernen."""
    prepared = records.copy()
    prepared[NUMERIC] = records[NUMERIC].apply(pd.to_numeric, errors="coerce")
    return prepared


prepared_batch = prepare_inputs(batch)
prepared_batch[NUMERIC].dtypes

In [ ]:
def find_base_issues(raw, prepared):
    """Trennt ungültige Messwerte von gültigen, aber ungewohnten Eingaben."""
    issues = []

    def add(mask, field, severity, reason):
        for request_id in raw.loc[mask, "request_id"]:
            issues.append((request_id, field, severity, reason))

    for column in NUMERIC:
        values = prepared[column]
        add(
            raw[column].notna() & values.isna(),
            column,
            "reject",
            f"{column}: not a number",
        )
        add(
            values.notna() & ~np.isfinite(values),
            column,
            "reject",
            f"{column}: not finite",
        )
        add(values.lt(0), column, "reject", f"{column}: negative value")
        bounds = model_info["numeric_training_ranges"][column]
        outside = values.lt(bounds["min"]) | values.gt(bounds["max"])
        add(outside, column, "review", f"{column}: outside the observed training range")
    add(
        prepared["Age"].gt(120),
        "Age",
        "reject",
        "Age: exceeds the exercise limit of 120",
    )
    for column in ["SibSp", "Parch"]:
        values = prepared[column]
        fractional = np.isfinite(values) & values.where(np.isfinite(values)).mod(1).ne(
            0
        )
        add(fractional, column, "reject", f"{column}: a count must be a whole number")
    for column in CATEGORICAL:
        unknown = raw[column].notna() & ~raw[column].isin(KNOWN_CATEGORIES[column])
        add(unknown, column, "review", f"{column}: category absent from training")
    add(
        raw[FEATURES].isna().all(axis=1),
        "all",
        "review",
        "All feature values are missing",
    )
    return pd.DataFrame(issues, columns=["request_id", "field", "severity", "reason"])


issues = find_base_issues(batch, prepared_batch)
issues

Der Encoder verwendet `handle_unknown="ignore"`: Eine ungewohnte Kategorie wird für dieses kategoriale Feature zu einem Block aus lauter Nullen. Er lernt dabei keine neue Kategorie und beweist auch nicht, dass die Vorhersage zuverlässig ist. Wir behalten eine Prüfmarkierung bei, auch wenn die Pipeline eine Ausgabe liefern kann.

Abgelehnte Zeilen können zusätzlich Prüfgründe haben; die Ablehnung hat Vorrang. Die Prüfung des Wertebereichs nutzt Kennzahlen der Trainingsdaten aus den gespeicherten Metadaten, keine neuen Kennzahlen des Batches.

[Referenz: OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

### Aufgabe 3: Regel hinzufügen und einen Schemafehler untersuchen

Erweitere den Validator um eine Regel deiner Wahl. Markiere zum Beispiel Zeilen, in denen mindestens drei der sechs Features fehlen, oder markiere leere oder nur aus Leerzeichen bestehende Kategorie-Strings, statt sie als neue Kategorienamen zu behandeln. Behalt die vorhandenen Prüfungen bei.

Erstell mindestens zwei zusätzliche Fälle, um deine Regel zu demonstrieren, und erklär für jeden Fall, ob die Regel ihn ablehnen oder markieren soll. Zeig außerdem die strukturelle Ablehnung einer doppelten `request_id` oder einer unerwarteten Spalte `Survived`. Diese Prüfungen gelten dem Eingabe-Workflow; sie sind keine neuen Evaluationen der Modellleistung.

<details>
<summary>Tipp für eine Regel zu fehlenden Features</summary>

Zähl mit `raw[FEATURES].isna().sum(axis=1)` die fehlenden Messwerte. Füg für deinen gewählten Schwellenwert einen Prüfgrund hinzu und untersuch dann Überschneidungen mit vorhandenen Gründen. Der Schwellenwert ist deine explizite Schnittstellenrichtlinie, kein gelernter Modellparameter.
</details>

In [ ]:
def find_issues(raw, prepared):
    """Behält die angeleiteten Prüfungen bei und markiert Requests, denen mindestens drei Features fehlen."""
    base_issues = find_base_issues(raw, prepared)
    many_missing = raw[FEATURES].isna().sum(axis=1).ge(3)
    extra_issues = raw.loc[many_missing, ["request_id"]].assign(
        field="all",
        severity="review",
        reason="At least three feature values are missing",
    )
    return pd.concat([base_issues, extra_issues], ignore_index=True)


# Einen Fall am gewählten Schwellenwert und einen knapp darunter hinzufügen.
rule_cases = pd.concat([new_passenger] * 2, ignore_index=True)
rule_cases.insert(0, "request_id", ["own-05", "own-06"])
rule_cases[["Age", "Fare"]] = np.nan
rule_cases.loc[0, "Embarked"] = np.nan
own_batch = pd.concat([own_requests, rule_cases], ignore_index=True)
find_issues(own_batch, prepare_inputs(own_batch))

Fehlen drei Features, führt das zu einer Prüfung, nicht zu einer Ablehnung: Die Pipeline kann sie imputieren, aber ein großer Teil des Eintrags wird rekonstruiert. Dieser Schwellenwert ist eine explizite Richtlinie für diese Aufgabe, kein validierter Schwellenwert für die Modellqualität. Den Fall mit zwei fehlenden Features markiert die neue Regel nicht.

Die ursprüngliche Prüffunktion heißt in dieser Lösung `find_base_issues`, damit ihre Erweiterung die vorhandenen Regeln nicht verwirft. `predict_batch` unten ruft das erweiterte `find_issues` auf.

In [ ]:
# Schemafehler separat demonstrieren; diese fehlerhaften Batches nicht zu own_batch hinzufügen.
broken_batches = {
    "Duplicate request ID": own_requests.assign(request_id="same-id"),
    "Unexpected target column": own_requests.assign(Survived=0),
}
for case_name, broken_batch in broken_batches.items():
    try:
        check_structure(broken_batch)
    except ValueError as error:
        print(f"{case_name}: {error}")

## 5. Batch vorhersagen und jeden Request behalten

Für die Vorhersage werden nur Zeilen ohne Ablehnungsgründe verwendet. Prüffälle behalten ihre Vorhersagen zur Kontrolle, wobei ihr Status und ihre Gründe deutlich sichtbar sind. Abgelehnte Zeilen behalten ihre Kennungen, haben aber keine Vorhersage.

Die Ausgabe muss für jeden Request eine Zeile behalten, in der ursprünglichen Reihenfolge. Lass nie stillschweigend schwierige Einträge weg, um den Rest dann als vollständigen Batch auszugeben.

In [ ]:
def predict_batch(records):
    """Validiert die Requests und gibt Vorhersagen mit einem expliziten Status pro Zeile zurück."""
    check_structure(records)
    # Die Request-IDs bewahren die Identität; vor dem Zuweisen der Ausgaben die pandas-Indizes zurücksetzen.
    records = records.reset_index(drop=True)
    prepared = prepare_inputs(records)
    found_issues = find_issues(records, prepared)
    rejected_ids = found_issues.loc[found_issues["severity"].eq("reject"), "request_id"]
    review_ids = found_issues.loc[found_issues["severity"].eq("review"), "request_id"]

    result = records[["request_id"]].copy()
    result["status"] = "predicted"
    result.loc[result["request_id"].isin(review_ids), "status"] = "review"
    result.loc[result["request_id"].isin(rejected_ids), "status"] = "rejected"
    result["predicted_class"] = pd.Series(pd.NA, index=result.index, dtype="Int64")
    result["probability_class_1"] = np.nan
    accepted = ~result["status"].eq("rejected")
    if accepted.any():
        inputs = prepared.loc[accepted, FEATURES]
        result.loc[accepted, "predicted_class"] = model.predict(inputs)
        result.loc[accepted, "probability_class_1"] = model.predict_proba(inputs)[
            :, positive_column
        ]
    reasons = found_issues.groupby("request_id")["reason"].agg("; ".join)
    result["notes"] = result["request_id"].map(reasons).fillna("")
    return result, found_issues


results, issues = predict_batch(batch)
results

In [ ]:
status_counts = (
    results["status"]
    .value_counts()
    .reindex(["predicted", "review", "rejected"], fill_value=0)
)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(
    status_counts.index, status_counts.values, color=["#398454", "#c18a25", "#b44545"]
)
ax.set(title="Input handling for the invented batch", ylabel="Requests")
fig.tight_layout()

### Aufgabe 4: Eingabequalität berichten

Bau eine kompakte Zusammenfassung mit der Gesamtzahl der Requests, der Anzahl je Status, dem Prozentsatz der Requests mit mindestens einem fehlenden Feature und den häufigsten Problemgründen. Zähl die betroffenen Requests, nicht die Problemeinträge: Eine einzelne Zeile kann mehrere Probleme haben.

Vergleich deine Ergebnisse mit deiner Triage-Tabelle aus Aufgabe 2 und erklär jede Abweichung. Nenn eine zusätzliche Prüfung, die dieser Workflow allein mit diesen Spalten nicht leisten kann, zum Beispiel die Prüfung der Einheiten oder der Herkunft eines plausibel wirkenden Werts.

In [ ]:
input_summary = pd.Series(
    {
        "requests": len(batch),
        **status_counts.to_dict(),
        "requests_with_missing_features_pct": 100
        * batch[FEATURES].isna().any(axis=1).mean(),
    },
    name="Input quality",
)
print(input_summary.to_string())
# Unterschiedliche betroffene Requests je Grund zählen, nicht wiederholte Problemeinträge.
reason_counts = (
    issues.groupby("reason")["request_id"].nunique().sort_values(ascending=False)
)
print("\nAffected requests per reason:")
print(reason_counts.to_string())
triage_comparison = expected_triage.merge(
    results[["request_id", "status"]], on="request_id", validate="one_to_one"
)
triage_comparison["matches"] = triage_comparison["expected_status"].eq(
    triage_comparison["status"]
)
triage_comparison[["request_id", "expected_status", "status", "matches"]]

Für den bereitgestellten Batch liefert die Musterlösung 5 gewöhnliche Vorhersagen, 5 Prüffälle und 4 Ablehnungen. Vier Requests haben mindestens ein fehlendes Feature (etwa 28,6%). Die Anzahlen der Probleme lassen sich nicht addieren: Ein Request kann mehrere Gründe haben.

Die neue Regel fügt dem Eintrag, in dem alle Werte fehlen, einen zweiten Prüfgrund hinzu, ohne seinen Status zu ändern. Plausibel wirkende Werte können trotzdem die falschen Einheiten haben oder aus einer unzuverlässigen Quelle stammen; das können diese Prüfungen allein anhand der sechs Spalten nicht feststellen.

## 6. Wahrscheinlichkeiten zusammen mit dem Eingabestatus lesen

Eine Vorhersage mit einer Wahrscheinlichkeit von 0,9 kann trotzdem von einer ungewohnten Kategorie oder einer Eingabe außerhalb des Wertebereichs der Trainingsdaten stammen. Die Wahrscheinlichkeit beschreibt die Ausgabe des trainierten Klassifikators; der Status beschreibt die Eingabequalität gemäß unserem Vertrag. Sie beantworten unterschiedliche Fragen.

In [ ]:
scored = results.loc[results["status"].ne("rejected")]
fig, ax = plt.subplots(figsize=(8, 4))
colors = scored["status"].map({"predicted": "#398454", "review": "#c18a25"})
ax.barh(scored["request_id"], scored["probability_class_1"], color=colors)
ax.axvline(0.5, color="gray", linestyle="--", label="Equal class probability")
ax.set(
    xlim=(0, 1),
    xlabel="Estimated P(Survived = 1)",
    title="Invented requests: probability and input status",
)
ax.legend()
fig.tight_layout()

Diskutiere: Kann die Zeile, in der alle Features fehlen, trotzdem eine Vorhersage erhalten? Warum? Welche Belege bräuchtest du, bevor du die angezeigten Wahrscheinlichkeiten als gut kalibriert bezeichnest?

<details>
<summary>Erklärung anzeigen</summary>

Die gefitteten Imputer ersetzen die fehlenden Werte, und der Klassifikator erstellt seine Vorhersage auf dieser konstruierten Darstellung. Er hat keine Informationen über diese neue Person gewonnen. Bei der Kalibrierung geht es um die Übereinstimmung zwischen Wahrscheinlichkeiten und beobachteten Ergebnissen über Beispiele mit Labels hinweg; dieser Batch ohne Labels kann sie nicht nachweisen. Lass den Prüfstatus und das Fehlen von Belegen zur Leistung sichtbar.
</details>

## 7. Empfindlichkeit ohne neues Training untersuchen

Halt fünf Features konstant und variiere das Alter innerhalb eines gewählten Wertebereichs. Das zeigt, wie das festgeschriebene Modell auf die Änderung einer Eingabe reagiert. Es ist kein kausales Experiment: Das Alter in diesem konstruierten Eintrag zu ändern, beweist nicht, dass das Alter eine Änderung beim Überleben verursacht.

In [ ]:
ages = np.arange(5, 81, 5)
sensitivity = pd.concat([new_passenger] * len(ages), ignore_index=True)
sensitivity["Age"] = ages
sensitivity_probability = model.predict_proba(sensitivity)[:, positive_column]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(ages, sensitivity_probability, marker="o", color="#4076a6")
ax.set(
    xlabel="Age in the constructed record",
    ylabel="Estimated P(Survived = 1)",
    ylim=(0, 1),
)
fig.tight_layout()

### Aufgabe 5: zwei Profile vergleichen

Wähl zwei erfundene Profile und variiere bei beiden dasselbe numerische Feature, wobei alle anderen Eingaben innerhalb jedes Profils konstant bleiben. Stell beide Wahrscheinlichkeitskurven dar. Du kannst das Alter oder den Fahrpreis untersuchen; dokumentiere den Wertebereich und finde alle Werte außerhalb des gespeicherten Wertebereichs der Trainingsdaten.

Beschreib die Unterschiede, abrupte Änderungen und eine Schlussfolgerung, die der Plot **nicht** rechtfertigt. Erklär, warum es keine bessere Leistung belegen würde, ein Modell oder einen Schwellenwert so zu tunen, dass diese erfundenen Beispiele schöner aussehen.

In [ ]:
profiles = {
    "Profile A: male": new_passenger.iloc[0].to_dict(),
    "Profile B: female": {**new_passenger.iloc[0].to_dict(), "Sex": "female"},
}
age_bounds = model_info["numeric_training_ranges"]["Age"]
sensitivity_rows = []
fig, ax = plt.subplots(figsize=(7, 3))
for name, profile in profiles.items():
    inputs = pd.DataFrame([profile] * len(ages))
    inputs["Age"] = ages
    probabilities = model.predict_proba(inputs)[:, positive_column]
    outside = inputs["Age"].lt(age_bounds["min"]) | inputs["Age"].gt(age_bounds["max"])
    sensitivity_rows.append(
        {
            "profile": name,
            "varied_feature": "Age",
            "range": f"{ages.min()} to {ages.max()}",
            "values_outside_training_range": int(outside.sum()),
            "min_probability": float(probabilities.min()),
            "max_probability": float(probabilities.max()),
            "probability_changes": int(np.count_nonzero(np.diff(probabilities))),
        }
    )
    ax.plot(ages, probabilities, marker="o", label=name)
ax.set(
    xlabel="Age in the constructed record",
    ylabel="Estimated P(Survived = 1)",
    title="Two constructed profiles: sensitivity to age",
    ylim=(0, 1),
)
ax.legend()
fig.tight_layout()
pd.DataFrame(sensitivity_rows)

Innerhalb jedes Profils ändert sich nur das Alter. Zwischen den Profilen unterscheidet sich nur die Kategorie des erfassten Geschlechts. Vergleich die angezeigten Wertebereiche der Wahrscheinlichkeiten und die Anzahl der Änderungen zwischen aufeinanderfolgenden Alterswerten: Sie fassen zusammen, wie sich die Reaktionen unterscheiden. Abrupte Sprünge spiegeln wider, welche Nachbarn aus den Trainingsdaten zur Stimmabgabe des KNN beitragen, keine kausalen Effekte oder Garantien für neue Reisende.

Die Tabelle weist Werte außerhalb des gespeicherten Wertebereichs der Trainingsdaten aus. Selbst eine konstruierte Kombination innerhalb des Wertebereichs ist nicht unbedingt gut in den Trainingsdaten vertreten. Das Modell so zu tunen, dass diese Kurven ohne Labels glatter aussehen, würde keine bessere Leistung belegen.

## 8. Ergebnisse exportieren und Übergabe vorbereiten

Exportiere die vollständige Ergebnistabelle, einschließlich der abgelehnten Requests. Eine separate Problemtabelle bewahrt die Details auf Ebene der Gründe. Die Metadaten des Modells dokumentieren eine frühere Evaluation; der Batch-Report dokumentiert die heutige Verarbeitung. Erfinde keine Accuracy für diesen Batch ohne Labels.

In [ ]:
results.to_csv(ARTIFACT_DIR / "titanic_predictions.csv", index=False)
issues.to_csv(ARTIFACT_DIR / "titanic_input_issues.csv", index=False)
batch_report = {
    "input_file": "data/titanic_prediction_requests.csv",
    "model_file": MODEL_PATH.name,
    "request_count": len(batch),
    "status_counts": {name: int(count) for name, count in status_counts.items()},
    "requests_with_missing_features": int(batch[FEATURES].isna().any(axis=1).sum()),
    "ground_truth_available": False,
}
(ARTIFACT_DIR / "titanic_batch_report.json").write_text(
    json.dumps(batch_report, indent=2) + "\n"
)
pd.Series(batch_report, name="Batch report")

### Aufgabe 6: eigenen Batch und eigene Übergabe abgeben

Verarbeite die Einträge, die du in den Aufgaben 1 und 3 erstellt hast, mit deinem erweiterten Validator. Exportiere ihre Vorhersagen, Probleme und einen Report mit ihren tatsächlichen Anzahlen und einer Beschreibung der Quelle. Speichere eine kurze `artifacts/model_handover.md` mit folgendem Inhalt:

1. Der Zweck und die historische Zielvariable des Modells.
2. Die sechs rohen Eingaben und wie fehlende, ungültige und unbekannte Werte behandelt werden.
3. Welche festgehaltenen Scores aus der CV und welche aus dem abschließenden Test stammen.
4. Deine zusätzliche Validierungsregel und der Grund für ihren Schweregrad.
5. Zwei Grenzen, darunter die fehlenden tatsächlichen Labels für deine neuen Beispiele.
6. Deine Empfehlung für den Umgang mit Prüffällen und welche Belege für einen echten Anwendungsfall nötig sind.

Diese Musterlösung verwendet Dateinamen mit dem Präfix `solution_`, darunter `artifacts/solution_model_handover.md`, damit sie deine eigene Übergabe nicht überschreibt.

Starte zum Schluss den Kernel neu und führ dieses Notebook noch einmal aus, um zu prüfen, dass sich die gespeicherte Pipeline laden lässt und die bereitgestellten Beispiele ohne die Variablen von Notebook 1 im Arbeitsspeicher funktionieren. Führ auch deine eigenen Aufgabenzellen erneut aus. Bestätige, dass dieselben gewöhnlichen Einträge vor und nach dem Speichern/Laden dieselben Vorhersagen erhalten.

In [ ]:
own_results, own_issues = predict_batch(own_batch)
own_status_counts = (
    own_results["status"]
    .value_counts()
    .reindex(["predicted", "review", "rejected"], fill_value=0)
)
own_report = {
    "source": "Six invented requests from Exercises 1 and 3 in this solution",
    "model_file": MODEL_PATH.name,
    "request_count": len(own_batch),
    "status_counts": {name: int(count) for name, count in own_status_counts.items()},
    "requests_with_missing_features": int(own_batch[FEATURES].isna().any(axis=1).sum()),
    "additional_rule": "Review requests missing at least three features",
    "ground_truth_available": False,
}
# Eigene Dateinamen verwenden, damit die Exporte der angeleiteten Beispiele erhalten bleiben.
own_results.to_csv(ARTIFACT_DIR / "solution_predictions.csv", index=False)
own_issues.to_csv(ARTIFACT_DIR / "solution_input_issues.csv", index=False)
(ARTIFACT_DIR / "solution_batch_report.json").write_text(
    json.dumps(own_report, indent=2) + "\n", encoding="utf-8"
)
own_results

Die sechs Requests bleiben in ihrer ursprünglichen Reihenfolge: Drei sind gewöhnliche Vorhersagen, zwei brauchen eine Prüfung, und einer wird ohne Vorhersage abgelehnt. Der exportierte Report dokumentiert diesen Batch, nicht das bereitgestellte Beispiel mit 14 Einträgen. Der Request mit dem unbekannten Hafen und der Request mit drei fehlenden Features bleiben identifizierbar.

In [ ]:
handover = f"""# Titanic Model Handover

- Purpose: demonstrate inference for the historical binary Survived label, not a real safety decision.
- Inputs: {", ".join(FEATURES)}. Preserve the training CSV's units and meanings.
- Handling: fitted imputers fill missing measurements; invalid values are rejected.
  Unfamiliar categories or values outside training ranges are flagged for review.
- Recorded training CV accuracy: {model_info["evaluation"]["cv_accuracy"]:.3f}.
  Recorded final test accuracy: {model_info["evaluation"]["test_accuracy"]:.3f}.
  These scores came from notebook 1, not these requests.
- Additional rule: review requests missing at least three features, since much of
  their representation is imputed. This is an explicit interface policy.
- Batch: {len(own_batch)} invented requests; no ground-truth outcomes are available.
- Limits: historical population coverage; uncalibrated KNN probabilities;
  plausible inputs can still be unrepresentative or have incorrect units.
- Recommendation: retain review flags and reasons; investigate before relying
  on flagged predictions. Real use needs labeled representative evaluation data,
  a defined decision context and error costs, and appropriate governance.
"""
(ARTIFACT_DIR / "solution_model_handover.md").write_text(handover, encoding="utf-8")
print(handover)

### Gespeichertes Modell in einem frisch gestarteten Kernel prüfen

Starte den Kernel neu und führ diese Lösung von oben aus. Sie erstellt dieselben erfundenen Requests neu und lädt das Artefakt ohne die Variablen von Notebook 1. Der kleine Vergleich unten prüft außerdem, dass erneutes Laden derselben Datei die Klassenvorhersagen und Wahrscheinlichkeiten beibehält; er ist keine Evaluation der Accuracy.

In [ ]:
reloaded_model = joblib.load(MODEL_PATH)
before = model.predict(ordinary_input)
after = reloaded_model.predict(ordinary_input)
before_probabilities = model.predict_proba(ordinary_input)
after_probabilities = reloaded_model.predict_proba(ordinary_input)
pd.Series(
    {
        "Same classes after loading": np.array_equal(before, after),
        "Same probabilities after loading": np.allclose(
            before_probabilities, after_probabilities
        ),
    },
    name="Saved-model consistency",
)

## Checkliste zum Abschluss

- Die vollständige Pipeline lässt sich in einem frisch gestarteten Kernel von der Festplatte laden; hier findet kein Training statt.
- Deine eigenen Beispiele enthalten gewöhnliche, fehlende, ungewohnte und ungültige Eingaben.
- Deine zusätzliche Validierungsregel ist erklärt und demonstriert.
- Jeder Request hat eine Ausgabezeile; abgelehnte Requests haben keine Vorhersage.
- Deine Batch-Zusammenfassung zählt die Requests korrekt und unterscheidet Eingabequalität von Modellleistung.
- Dein Plot mit zwei Profilen ist ohne kausale Aussage interpretiert.
- Die exportierten Dateien beschreiben deinen tatsächlichen Batch, und die Übergabe erklärt die Grenzen.

### Optionale Erweiterungen

- Mach aus deinem validierten Batch-Workflow ein kleines Skript `predict.py` mit Eingabe- und Ausgabepfaden. Lass das gespeicherte Modell unverändert.
- Vergleich CSV-Vorhersagen für den ganzen Batch und in Chunks; prüf die Reihenfolge der Ausgabe und die Eindeutigkeit der Kennungen über die Chunks hinweg.
- Tausch einen erfundenen Eingabe-Batch mit einer anderen lernenden Person und erklär jede Ablehnung oder Prüfmarkierung.